<a href="https://colab.research.google.com/github/emmanuelmassawe/fine-tunning-llm-with-lora-and-qlora/blob/main/fine_tunning_llm_with_lora_and_qlora_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#this is llm fine-tuning using lora and qlora

##install necessary packages

In [1]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


##import neccesary libraries

In [2]:
import gc
import os

import torch
from datasets import load_dataset
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

In [3]:
OUTPUT_DIR = f"/content/drive/MyDrive/out-{MODE}"

In [4]:
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    DTYPE = torch.bfloat16   # Ampere na kuendelea (A100, L4, RTX 30xx)
else:
    DTYPE = torch.float16    # T4

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
## CONFIG: badilisha hapa tu
MODE = "qlora"                       # "lora" (16-bit) au "qlora" (4-bit)
MODEL_NAME = "Qwen/Qwen2.5-0.5B"     # model ndogo kwa majaribio
DATASET = "yahma/alpaca-cleaned"     # dataset ya Hugging Face
DATA_FILE = None                     # mfano: "data.json" ukitumia faili lako (inapita DATASET)
MAX_SAMPLES = 1000
MAX_LENGTH = 512
EPOCHS = 1
LR = 0.01
BATCH_SIZE = 2
GRAD_ACCUM = 8
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MERGE_AFTER_TRAINING = False

In [14]:
# T4 haina bf16, kwa hiyo tunachagua kiotomatiki
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float16

In [15]:
# ---------------------------------------------------------------
# HATUA 1: Pakia model (hapa ndipo quantization inapotokea)
# ---------------------------------------------------------------
def load_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if MODE == "qlora":
        # Uzito wa model unahifadhiwa kwa biti 4
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",         # NormalFloat4
            bnb_4bit_use_double_quant=True,    # quantize hata constants
            bnb_4bit_compute_dtype=DTYPE,      # hesabu kwa 16-bit
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=bnb_config, device_map="auto"
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, torch_dtype=DTYPE, device_map="auto"
        )
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()

    model.config.use_cache = False
    return model, tokenizer


In [16]:
# ---------------------------------------------------------------
# HATUA 2: Weka adapter za LoRA
# ---------------------------------------------------------------
def add_lora(model):
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules="all-linear",
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    # Parameters za LoRA ziwe fp32 ili fp16 training isipate hitilafu
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    model.print_trainable_parameters()
    return model

In [17]:
# ---------------------------------------------------------------
# HATUA 3: Andaa dataset (safu: instruction, input (si lazima), output)
# ---------------------------------------------------------------
def build_dataset(tokenizer):
    if DATA_FILE:
        ds = load_dataset("json", data_files=DATA_FILE, split="train")
    else:
        ds = load_dataset(DATASET, split="train")

    ds = ds.shuffle(seed=42).select(range(min(MAX_SAMPLES, len(ds))))

    def format_example(ex):
        if ex.get("input"):
            prompt = (
                f"### Instruction:\n{ex['instruction']}\n\n"
                f"### Input:\n{ex['input']}\n\n### Response:\n"
            )
        else:
            prompt = f"### Instruction:\n{ex['instruction']}\n\n### Response:\n"
        text = prompt + ex["output"] + tokenizer.eos_token
        return tokenizer(text, truncation=True, max_length=MAX_LENGTH)

    ds = ds.map(format_example, remove_columns=ds.column_names)
    print("Idadi ya mifano:", len(ds))
    return ds

In [18]:
# ---------------------------------------------------------------
# HATUA 4: Train
# ---------------------------------------------------------------
def train(model, tokenizer, dataset):
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=5,
        logging_steps=10,
        save_strategy="epoch",
        bf16=(DTYPE == torch.bfloat16),
        fp16=(DTYPE == torch.float16),
        optim="paged_adamw_8bit" if MODE == "qlora" else "adamw_torch",
        gradient_checkpointing=True,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    trainer.train()

    # Hifadhi adapter tu (MB chache), si model nzima
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Adapter imehifadhiwa kwenye:", OUTPUT_DIR)

In [19]:
@torch.no_grad()
def quick_test(model, tokenizer):
    model.config.use_cache = True
    model.eval()
    prompt = "### Instruction:\nEleza maana ya quantization kwa ufupi.\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
    print(tokenizer.decode(out[0], skip_special_tokens=True))

In [12]:
def merge_adapter(tokenizer):
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=DTYPE, device_map="auto"
    )
    merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
    merged_dir = OUTPUT_DIR + "-merged"
    merged.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)
    print("Model iliyounganishwa imehifadhiwa kwenye:", merged_dir)


def main():
    if not torch.cuda.is_available():
        raise SystemExit("GPU haipo. Nenda Runtime -> Change runtime type -> T4 GPU.")
    if MODE not in ("lora", "qlora"):
        raise SystemExit('MODE lazima iwe "lora" au "qlora".')

    print(f"MODE={MODE}  DTYPE={DTYPE}  OUTPUT_DIR={OUTPUT_DIR}")

    model, tokenizer = load_model_and_tokenizer()
    model = add_lora(model)
    dataset = build_dataset(tokenizer)
    train(model, tokenizer, dataset)
    quick_test(model, tokenizer)

    if MERGE_AFTER_TRAINING:
        del model
        gc.collect()
        torch.cuda.empty_cache()
        merge_adapter(tokenizer)


main()

MODE=qlora  DTYPE=torch.bfloat16  OUTPUT_DIR=/content/drive/MyDrive/out-qlora


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (9 outputs left out)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Idadi ya mifano: 1000


Step,Training Loss
10,1.616278
20,1.540014
30,1.446237
40,1.408891
50,1.515859
60,1.431875


Adapter imehifadhiwa kwenye: /content/drive/MyDrive/out-qlora
### Instruction:
Eleza maana ya quantization kwa ufupi.

### Response:
Eleza maana ya quantization kwa ufupi. In this phrase, "Eleza maana ya quantization kwa ufupi" means "The teacher is teaching a new topic at an intermediate level." The word "maana" means "teacher," "quantization" refers to teaching, "kwa" means "at," and "ufupi" means "intermediate." So, the whole phrase means "The teacher is teaching a new topic at an intermediate level
